# Bubble Bi — Colab runner

Runs the full world-model pipeline (data → tokenizers → next-token predictor) on Colab.

**Pick a runtime first:** Runtime → Change runtime type → CPU / GPU (T4) / TPU.
The code detects it automatically. **GPU is the recommended target** (our models are small,
so a T4/A100 will typically beat a TPU here).


## 1. Mount Drive + get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/bubble_bi'
!mkdir -p {DRIVE}/artifacts/raw {DRIVE}/artifacts/cache
print('drive ready:', DRIVE)


In [ ]:
import os
REPO = 'https://github.com/hockper/Quant-AI-2026.git'
if os.path.isdir('/content/Quant-AI-2026'):
    !cd /content/Quant-AI-2026 && git pull --ff-only
else:
    !git clone {REPO} /content/Quant-AI-2026
%cd /content/Quant-AI-2026


In [ ]:
# NOTE: torch is deliberately NOT in requirements.txt, so Colab's
# pre-installed CUDA/XLA torch is preserved.
!pip install -q -r requirements.txt


## 2. Runtime check + tests (the migration's proof)

In [ ]:
import torch
from bubble_bi.runtime import detect_runtime, resolve_device

rt = detect_runtime()
print('torch  :', torch.__version__)
print('runtime:', rt)
print('device :', resolve_device('auto'))
if rt == 'cpu':
    print('\nTip: Runtime -> Change runtime type -> GPU for real speed.')


In [ ]:
# If these pass here, the port to Colab's Python/pandas is proven.
!python -m pytest -q


## 3. Config (points at Drive; override anything here)

In [ ]:
from bubble_bi.colab import load_colab_config

DRIVE = '/content/drive/MyDrive/bubble_bi'
cfg = load_colab_config('configs/m3.yaml', DRIVE)   # m3 config covers every stage
print('raw  :', cfg.data.raw_dir)
print('cache:', cfg.data.cache_dir)
print('device:', cfg.train.device)


## 4. Data (run once; cached on Drive)

In [ ]:
from bubble_bi.cli import run_ingest, build_panel_from_raw, run_baseline

run_ingest(cfg)                 # yfinance -> Drive parquet
panel = build_panel_from_raw(cfg)
print('panel:', panel.features.shape)
run_baseline(cfg)               # ridge RankIC floor


## 5. Staged tokenizer

Phase 1 TS → Phase 2 CS → Phase 3 fusion (frozen encoders).
Pass `resume=True` to continue a run a Colab disconnect killed.


In [ ]:
from bubble_bi.cli import train_tokenizer, train_cs, train_fusion

cfg.train.batch_size = 256      # scale up on GPU
cfg.train.max_steps = 2000

train_tokenizer(cfg, run_name='ts_gpu')      # add resume=True after a disconnect
train_cs(cfg, run_name='cs_gpu')
train_fusion(cfg, run_name='fusion_gpu')


In [ ]:
from bubble_bi.cli import eval_tokenizer, eval_cs, eval_fusion

eval_tokenizer(cfg, run_name='ts_gpu')
eval_cs(cfg, run_name='cs_gpu')
eval_fusion(cfg, run_name='fusion_gpu')


## 6. Token grid + next-token predictor

In [ ]:
from bubble_bi.cli import tokenize_panel, train_predictor, eval_predictor

tokenize_panel(cfg)                            # frozen tokenizer -> tokens.npz
train_predictor(cfg, run_name='pred_gpu')      # resume=True to continue
eval_predictor(cfg, run_name='pred_gpu')


## 7. Plots

In [ ]:
from bubble_bi.cli import plot_metrics
from IPython.display import Image, display

for p in plot_metrics(cfg, ['pred_gpu']):
    display(Image(filename=p))


## 8. Fine-tuning loop

Change hyperparameters, give the run a NEW name, then overlay the curves:


In [ ]:
cfg2 = load_colab_config('configs/m3.yaml', DRIVE,
                         pred_layers=6, d_model=256, batch_size=256, max_steps=4000)
train_predictor(cfg2, run_name='pred_big')
eval_predictor(cfg2, run_name='pred_big')

plot_metrics(cfg, ['pred_gpu', 'pred_big'])   # overlaid comparison
